# CliniqAI — Router Stage 1 (gatekeeper) v4 — full retrain, BreakHis positive source

**Why v4 and not another fine-tune:** v3 attempts 1 and 2 fine-tuned from v2 with resolution-
degradation augmentation (p=0.5 then p=0.85) and both failed identically on the real-world
sanity image — resolution augmentation alone doesn't unlearn the "blocky texture -> medical"
shortcut once it's already baked into v2's weights. This time the actual training images for
`breast_histology` are being replaced (the corrected, deduped BreakHis data — 7,909 real images,
four real magnification levels, no more 50x50 patches), which is a bigger change than
augmentation can produce from a fine-tune starting point. Retraining from ImageNet weights (like
v2 originally was) rather than fine-tuning from v2 avoids carrying the old shortcut forward.

**Everything else matches the v2 recipe exactly** — same 6 Kaggle negative sources (already on
disk under `data/raw/gatekeeper_negatives_v2/`), same 12,000-per-modality positive cap, same
`fruits` held-out generalization test, same wide-head EfficientNet-B0 architecture. Only the
`breast_histology` positive source changed.

**Known Windows gotcha applied here too:** globbing `*.png` and `*.PNG` separately on Windows
double-counts every file (bit the breast_histology specialist notebook earlier this project) —
this notebook dedupes by resolved path everywhere it globs images, not just for BreakHis.

Run top to bottom. Needs `kaggle.json` configured only if any negative source folder is missing —
otherwise no fresh downloads needed, everything positive/negative is already on disk.

## Step 0 — Imports

In [ ]:
import os
import sys
import json
import time
import shutil
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())


## Step 1 — Config

In [ ]:
PROJECT_ROOT = Path(r"E:\programming\CliniqAI")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
NEGATIVES_DIR = DATA_RAW / "gatekeeper_negatives_v2"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODELS_DIR = PROJECT_ROOT / "models" / "router"
BACKEND_MODELS_DIR = PROJECT_ROOT / "backend" / "models" / "router"
SANITY_DIR = DATA_RAW / "manual_sanity_check"

# modality -> its positive-source folder under data/raw/
POSITIVE_SOURCES = {
    "bone_fracture": DATA_RAW / "bone_fracture",
    "brain_mri": DATA_RAW / "brain_tumor",
    "breast_histology": DATA_RAW / "breast_histology_v2",   # <-- the only thing that changed vs. v2
    "kidney_ct": DATA_RAW / "kidney_stone",
    "pneumonia": DATA_RAW / "pneumonia",
    "retinal": DATA_RAW / "diabetic_retinopathy",
    "skin_lesion": DATA_RAW / "skin_lesion",
}
POSITIVE_CAP = 12000

# negative source subfolders under gatekeeper_negatives_v2/ -- 'fruits' is held out entirely,
# never enters train/val/test, only used as a post-hoc generalization check (Step 12)
NEGATIVE_SOURCES = ["food101", "scenes", "natural_images", "imagenet_mini", "human_faces", "synthetic_screenshots"]
HELD_OUT_SOURCE = "fruits"

IMG_SIZE = 224
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
EARLY_STOP_PATIENCE = 8
MAX_EPOCHS = 30
SEED = 42
CHECKPOINT_NAME = "router_stage1_v4_efficientnet_b0"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_EXTS = ("*.png", "*.PNG", "*.jpg", "*.JPG", "*.jpeg", "*.JPEG")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

MODELS_DIR.mkdir(parents=True, exist_ok=True)
BACKEND_MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("device:", device)


## Step 2 — Glob helper (dedup-safe)

Same fix as the breast_histology specialist notebook: Windows globs are case-insensitive, so
matching several extension patterns and concatenating can double-count files. Dedupe by resolved
path everywhere.

In [ ]:
def glob_images(folder, exts=IMG_EXTS):
    seen = set()
    out = []
    for ext in exts:
        for p in folder.glob(f"**/{ext}"):
            rp = p.resolve()
            if rp not in seen:
                seen.add(rp)
                out.append(p)
    return out


## Step 3 — Index negative sources

Reuses what's already on disk under `gatekeeper_negatives_v2/` — no fresh download. Prints
counts so you can sanity-check against the v2 numbers documented in the project log rather than
assuming they still match (folders may have changed since).

In [ ]:
negative_records = []  # (path, "negative", source_tag)
for source in NEGATIVE_SOURCES:
    folder = NEGATIVES_DIR / source
    if not folder.exists():
        print(f"WARNING: {folder} not found, skipping {source}")
        continue
    imgs = glob_images(folder)
    for p in imgs:
        negative_records.append((str(p), "negative", source))
    print(f"{source}: {len(imgs)} images")

held_out_folder = NEGATIVES_DIR / HELD_OUT_SOURCE
held_out_images = glob_images(held_out_folder) if held_out_folder.exists() else []
print(f"\n{HELD_OUT_SOURCE} (held out, not used in train/val/test): {len(held_out_images)} images")
print(f"\nTotal negatives indexed: {len(negative_records)}")


## Step 4 — Index positive sources (7 modalities, capped 12,000 each)

`breast_histology` is the only source that changed vs. v2 — points at the corrected, deduped
BreakHis data instead of the old IDC 50x50 patches.

In [ ]:
positive_records = []  # (path, "medical", modality_tag)
rng = random.Random(SEED)

for modality, folder in POSITIVE_SOURCES.items():
    if not folder.exists():
        print(f"WARNING: {folder} not found, skipping {modality}")
        continue
    imgs = glob_images(folder)
    if len(imgs) > POSITIVE_CAP:
        imgs = rng.sample(imgs, POSITIVE_CAP)
    for p in imgs:
        positive_records.append((str(p), "medical", modality))
    print(f"{modality}: {len(imgs)} images (source: {folder.name})")

print(f"\nTotal positives indexed: {len(positive_records)}")


## Step 5 — Corrupted-file validation pass (project convention)

In [ ]:
all_records = negative_records + positive_records
bad_paths = set()
for path, *_ in all_records:
    try:
        with Image.open(path) as im:
            im.load()
    except Exception:
        bad_paths.add(path)

if bad_paths:
    print(f"Dropping {len(bad_paths)} corrupted/unreadable files.")
    all_records = [r for r in all_records if r[0] not in bad_paths]
else:
    print("No corrupted files found.")
print(f"{len(all_records)} usable images remain.")


## Step 6 — Per-source train/val/test split (80/10/10)

Matches the v2 approach: split independently per source_tag so every source is proportionally
represented in all three sets (not patient-grouped here — these aren't per-patient medical
datasets like BreakHis, they're general image classification sources plus specialist folders).

In [ ]:
by_source = defaultdict(list)
for r in all_records:
    by_source[r[2]].append(r)

train_records, val_records, test_records = [], [], []
for source, recs in by_source.items():
    recs = recs[:]
    random.Random(SEED).shuffle(recs)
    n = len(recs)
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    train_records.extend(recs[:n_train])
    val_records.extend(recs[n_train:n_train + n_val])
    test_records.extend(recs[n_train + n_val:])

random.Random(SEED).shuffle(train_records)
random.Random(SEED).shuffle(val_records)
random.Random(SEED).shuffle(test_records)

print(f"train: {len(train_records)} | val: {len(val_records)} | test: {len(test_records)}")
print(f"\nPer-source counts (train/val/test):")
for source in sorted(by_source.keys()):
    tr = sum(1 for r in train_records if r[2] == source)
    va = sum(1 for r in val_records if r[2] == source)
    te = sum(1 for r in test_records if r[2] == source)
    print(f"  {source}: {tr} / {va} / {te}")


## Step 7 — Write the standalone Dataset module (v4)

New file rather than overwriting `gatekeeper_dataset_v2.py` — that file's committed history
documents the two failed fine-tune attempts and is worth preserving as-is. Same class shape,
lighter optional resolution jitter (p=0.3) kept only as insurance, not as the primary fix this
time — the real fix is the corrected training data itself.

In [ ]:
dataset_module_path = SCRIPTS_DIR / "gatekeeper_dataset_v4.py"
dataset_module_code = '''"""
Standalone, importable Dataset for Router Stage 1 v4 training.
Must live in its own module (not a notebook cell) for Windows DataLoader multiprocessing
(num_workers > 0) to work -- worker subprocesses re-import this module fresh, so
ImageFile.LOAD_TRUNCATED_IMAGES must be set here at module level, not just in the notebook.
"""
import random
from pathlib import Path
from PIL import Image, ImageFile
from torch.utils.data import Dataset

ImageFile.LOAD_TRUNCATED_IMAGES = True


class GatekeeperDatasetV4(Dataset):
    """records: list of (path, label_str, source_tag). label_str in {\'medical\',\'negative\'}."""

    def __init__(self, records, label_to_idx, transform=None):
        self.records = records
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def _load(self, idx):
        path, label_str, _tag = self.records[idx]
        img = Image.open(path).convert("RGB")
        img.load()
        label = self.label_to_idx[label_str]
        return img, label

    def __getitem__(self, idx):
        try:
            img, label = self._load(idx)
        except (OSError, Image.DecompressionBombError):
            fallback_idx = (idx + 1) % len(self.records)
            img, label = self._load(fallback_idx)
        if self.transform:
            img = self.transform(img)
        return img, label


class RandomResolutionDegradation:
    """Randomly downsamples then upscales back. Applied only in the train transform.
    Kept as light insurance in v4 (p=0.3) -- not the primary fix, since the primary fix is
    the corrected breast_histology training data itself."""

    def __init__(self, p=0.3, low_res_range=(48, 128)):
        self.p = p
        self.low_res_range = low_res_range

    def __call__(self, img):
        if random.random() > self.p:
            return img
        w, h = img.size
        low = random.randint(*self.low_res_range)
        img = img.resize((low, low), Image.BILINEAR)
        img = img.resize((w, h), Image.BILINEAR)
        return img
'''

dataset_module_path.write_text(dataset_module_code, encoding="utf-8")
print(f"Wrote {dataset_module_path}")

sys.path.insert(0, str(SCRIPTS_DIR))
from gatekeeper_dataset_v4 import GatekeeperDatasetV4, RandomResolutionDegradation


## Step 8 — Labels, transforms

In [ ]:
class_names = ["non_medical", "medical"]
label_to_idx = {"negative": 0, "medical": 1}

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    RandomResolutionDegradation(p=0.3, low_res_range=(48, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


## Step 9 — Datasets & DataLoaders

In [ ]:
train_ds = GatekeeperDatasetV4(train_records, label_to_idx, transform=train_transform)
val_ds = GatekeeperDatasetV4(val_records, label_to_idx, transform=eval_transform)
test_ds = GatekeeperDatasetV4(test_records, label_to_idx, transform=eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")


## Step 10 — Model (wide head, matches both router stages' existing checkpoint format)

In [ ]:
def build_model(num_classes, hidden_dim=512):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, hidden_dim),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(hidden_dim, num_classes),
    )
    return model

model = build_model(len(class_names)).to(device, memory_format=torch.channels_last)
print(model.classifier)


## Step 11 — Loss, optimizer, scheduler, AMP

In [ ]:
train_label_counts = Counter(r[1] for r in train_records)
n_neg, n_pos = train_label_counts["negative"], train_label_counts["medical"]
class_weights = torch.tensor(
    [ (n_neg + n_pos) / (2 * n_neg), (n_neg + n_pos) / (2 * n_pos) ],
    device=device,
)
print("train label counts:", dict(train_label_counts), "| class weights:", class_weights.tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)


## Step 12 — Training loop

Per-epoch checkpointing on val_loss improvement, resume cell below for kernel-restart recovery.

In [ ]:
checkpoint_inprogress_path = MODELS_DIR / f"{CHECKPOINT_NAME}.checkpoint_inprogress.pt"
history = []
best_val_loss = float("inf")
epochs_no_improve = 0
start_epoch = 0

def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

for epoch in range(start_epoch, MAX_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    print(f"epoch {epoch+1:02d} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
          f"| val_loss {val_loss:.4f} acc {val_acc:.4f} | {elapsed:.1f}s")

    history.append({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "classes": class_names,
            "num_classes": len(class_names),
            "architecture": "efficientnet_b0",
            "epoch": epoch + 1,
            "best_val_loss": best_val_loss,
        }, checkpoint_inprogress_path)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs).")
            break

print("Training done. Best val_loss:", best_val_loss)


## Step 13 — Resume from checkpoint (only if the kernel was restarted mid-training)

Re-run Steps 0-11 first, then this cell instead of starting Step 12 from scratch.

In [ ]:
# resume_ckpt = torch.load(checkpoint_inprogress_path, map_location=device)
# model.load_state_dict(resume_ckpt["model_state_dict"])
# start_epoch = resume_ckpt["epoch"]
# best_val_loss = resume_ckpt["best_val_loss"]
# print(f"Resumed from epoch {start_epoch}, best_val_loss={best_val_loss:.4f}")
# # then re-run the Step 12 training loop cell, which will pick up from start_epoch


## Step 14 — Final save (project root + backend copy)

In [ ]:
best_ckpt = torch.load(checkpoint_inprogress_path, map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])

final_path = MODELS_DIR / f"{CHECKPOINT_NAME}.pt"
torch.save(best_ckpt, final_path)

backend_final_path = BACKEND_MODELS_DIR / f"{CHECKPOINT_NAME}.pt"
shutil.copy2(final_path, backend_final_path)

history_path = MODELS_DIR / f"{CHECKPOINT_NAME}.history.json"
history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

print("Saved:", final_path)
print("Saved:", backend_final_path)
print("Saved:", history_path)


## Step 15 — In-distribution test accuracy, per source

Reported for completeness, but per the project's own established finding (twice now, on the v3
attempts): this metric alone does NOT distinguish a real fix from a broken one, since every
source here is evaluated on the same distribution it trained on. Step 16 is the test that
actually matters.

In [ ]:
model.eval()
all_preds, all_labels, all_sources = [], [], []
idx = 0
with torch.no_grad():
    for images, labels in test_loader:
        batch_size = images.size(0)
        images = images.to(device, memory_format=torch.channels_last)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_sources.extend(r[2] for r in test_records[idx:idx + batch_size])
        idx += batch_size

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
overall_acc = (all_preds == all_labels).mean()
print(f"Overall in-distribution test accuracy: {overall_acc:.4f} (n={len(all_labels)})")
print("\nPer-source accuracy:")
for source in sorted(set(all_sources)):
    sidx = [i for i, s in enumerate(all_sources) if s == source]
    acc = (all_preds[sidx] == all_labels[sidx]).mean()
    print(f"  {source}: {acc:.4f} (n={len(sidx)})")


## Step 16 — Real-world sanity check (the test that actually matters)

Runs the crisp, full-resolution Wikimedia breast_histology image (and anything else you've
dropped in `manual_sanity_check/`) with NO manual degradation this time — the whole point of v4
is that it shouldn't need degrading anymore. If this still says `non_medical`, the data swap
alone wasn't sufficient and the zoom/crop-scale hypothesis needs revisiting.

In [ ]:
sanity_images = glob_images(SANITY_DIR)
print(f"Found {len(sanity_images)} sanity images in {SANITY_DIR}")

model.eval()
for img_path in sanity_images:
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"{img_path.name}: failed to open ({e})")
        continue
    x = eval_transform(img).unsqueeze(0).to(device, memory_format=torch.channels_last)
    with torch.no_grad():
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            probs = torch.softmax(model(x), dim=1)[0]
    pred_idx = probs.argmax().item()
    print(f"\n{img_path.name}:")
    for i, c in enumerate(class_names):
        marker = " <-- predicted" if i == pred_idx else ""
        print(f"  {c}: {probs[i].item():.4f}{marker}")


## Step 17 — Generalization check on held-out `fruits` (never trained on)

Confirms the fix doesn't overcorrect into "everything is medical" — same discipline as v2's
original held-out generalization test.

In [ ]:
if held_out_images:
    model.eval()
    correct = 0
    sample = held_out_images[:500] if len(held_out_images) > 500 else held_out_images
    with torch.no_grad():
        for img_path in sample:
            try:
                img = Image.open(img_path).convert("RGB")
            except Exception:
                continue
            x = eval_transform(img).unsqueeze(0).to(device, memory_format=torch.channels_last)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                probs = torch.softmax(model(x), dim=1)[0]
            pred = probs.argmax().item()
            if class_names[pred] == "non_medical":
                correct += 1
    print(f"fruits (held out): {correct}/{len(sample)} correctly classified non_medical ({correct/len(sample):.4f})")
else:
    print("No held-out fruits images found -- skipping generalization check.")


## Notes

- **Do not point `backend/app/config.py`'s `ROUTER_STAGE1_PATH` at v4 until Step 16 passes** —
  same discipline as v3: in-distribution accuracy (Step 15) is not sufficient evidence on its
  own, real-world sanity is.
- If Step 16 still rejects the crisp Wikimedia image: the data swap wasn't sufficient by itself,
  and the next thing to try is the crop-scale diagnostic from the original gatekeeper bug
  writeup (crop the sanity image to patch-scale, see if that flips the prediction) — now against
  a model that's never seen resolution-degraded-only data, which is a cleaner test than running
  it against a v3 checkpoint.
- If Step 16 passes but Step 17 (fruits generalization) drops noticeably below v2's baseline:
  the fix may have overcorrected — worth widening the sanity-image set (a few more real
  non-medical photos) before deploying.
- Worth committing `scripts/gatekeeper_dataset_v4.py` and this notebook to `feature/router` once
  training succeeds, whichever way Step 16 comes out — negative results are still worth keeping,
  same as the v3 attempts.
- `models/router/router_stage1_v4_efficientnet_b0.pt` (and the backend copy) will need uploading
  to the shared Google Drive once validated, same as the other checkpoints.